# Preprocessing — Jalur **IndoBERT**

Notebook **self-contained** (tanpa import `src/`), bisa lokal maupun Colab.

IndoBERT memakai **cleaning minimal** karena modelnya sudah paham bahasa Indonesia
natural + punya tokenizer subword sendiri. **Stemming/stopword/normalisasi slang
TIDAK dilakukan** — imbuhan, kata fungsi, tanda baca, dan negasi ("tidak", "bukan")
justru penting untuk konteks sentimen. Hanya 1 tahap:
1. **clean_for_bert** — lowercase, buang URL/mention/emoji; tanda baca & angka
   dipertahankan.

> **Split identik** dengan notebook SVM: urut `comment_id` + `seed=42`, split SEBELUM
> preprocessing → test set kedua model persis sama (perbandingan adil).

## 0. Install dependency

In [ ]:
!pip -q install pyarrow scikit-learn

## 1. Muat data berlabel (balanced 3.000)

In [ ]:
import os, pandas as pd

# Path lokal default; kalau tak ada (mis. di Colab) -> minta upload.
INPUT = "outputs/labeling/balanced_1000.csv"
if not os.path.exists(INPUT):
    try:
        from google.colab import files
        up = files.upload()
        INPUT = next(iter(up))
    except Exception:
        raise FileNotFoundError(
            "balanced_1000.csv tidak ditemukan. Taruh di folder kerja "
            "atau upload saat diminta (Colab)."
        )

df = pd.read_csv(INPUT, encoding="utf-8-sig")
LABELS = ["Negatif", "Netral", "Positif"]          # urutan menentukan id: Neg=0, Net=1, Pos=2
df = df[df["label"].isin(LABELS)].copy()
print(f"{len(df)} baris berlabel dimuat dari {INPUT}")
print(df["label"].value_counts().reindex(LABELS).to_string())

## 2. Fungsi preprocessing IndoBERT (inline)

In [ ]:
import re
from typing import List

_RE_URL = re.compile('https?://\\S+|www\\.\\S+', re.IGNORECASE)
_RE_MENTION = re.compile('@\\w+')
_RE_EMOJI = re.compile('[😀-🙏🌀-🗿🚀-\U0001f6ff🜀-🝿🞀-\U0001f7ff🠀-\U0001f8ff🤀-🧿🨀-\U0001fa6f🩰-\U0001faff✂-➰Ⓜ-🉑🤦-🤷\u200d♀-♂☀-⭕⏏⏩⌚️〰]+', re.UNICODE)
_RE_MULTI_SPACE = re.compile('\\s+')

def clean_for_bert(text: str) -> str:
    """Cleaning minimal untuk jalur IndoBERT (morfologi terjaga)."""
    if not text or not isinstance(text, str):
        return ""
    t = text.lower()
    t = _RE_URL.sub(" ", t)
    t = _RE_MENTION.sub(" ", t)
    t = _RE_EMOJI.sub(" ", t)
    t = re.sub(r"[^\w\s.,!?;:'\"()-]", " ", t)
    t = _RE_MULTI_SPACE.sub(" ", t)
    return t.strip()


def make_text_bert(text: str) -> str:
    """Pipeline final jalur IndoBERT: cleaning minimal saja."""
    return clean_for_bert(text or "")

## 3. Lihat transformasi (1 tahap)

In [ ]:
def trace_bert(text: str) -> None:
    print(f"RAW              : {text!r}")
    print(f"clean_for_bert   : {clean_for_bert(text)!r}   <- FINAL (morfologi & tanda baca utuh)")

trace_bert("Persidangan belum dilaksanakan kok sdh ngajuin ahli, kalo gini rakyat dibikin binggung")

## 4. Terapkan ke semua data + split identik (70/20/10)

In [ ]:
from sklearn.model_selection import train_test_split

LABEL2ID = {lab: i for i, lab in enumerate(LABELS)}

# --- SPLIT IDENTIK utk SVM & IndoBERT ---
# Urutkan comment_id + seed tetap -> kedua notebook menghasilkan train/val/test
# yang PERSIS SAMA (test set sama -> perbandingan adil). Split dilakukan SEBELUM
# preprocessing, jadi tak terpengaruh perbedaan cleaning antar jalur.
df = df.sort_values("comment_id").reset_index(drop=True)
df["label_id"] = df["label"].map(LABEL2ID)

SEED = 42
tmp, df_test = train_test_split(df, test_size=0.10, stratify=df["label_id"], random_state=SEED)
df_train, df_val = train_test_split(tmp, test_size=0.20/0.90, stratify=tmp["label_id"], random_state=SEED)

splits = {}
for name, part in [("train", df_train), ("val", df_val), ("test", df_test)]:
    part = part.copy()
    part["bert"] = part["text"].astype(str).map(make_text_bert)      # preprocessing jalur ini
    before = len(part)
    part = part[part["bert"].str.len() > 0]                          # buang yang kosong stlh cleaning
    if before - len(part):
        print(f"[{name}] {before-len(part)} baris dibuang (kosong stlh preprocessing)")
    splits[name] = part.reset_index(drop=True)

for name, part in splits.items():
    dist = " ".join(f"{k}={v}" for k, v in part["label"].value_counts().reindex(LABELS).items())
    print(f"[{name}] n={len(part)} | {dist}")

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 60)
splits["train"][["text", "bert", "label"]].head(8)

## 5. Simpan split siap-model

In [ ]:
OUTDIR = "data/processed/indobert"
os.makedirs(OUTDIR, exist_ok=True)
cols = ["comment_id", "video_id", "text", "bert", "label", "label_id"]
for name, part in splits.items():
    keep = [c for c in cols if c in part.columns]
    part[keep].to_parquet(f"{OUTDIR}/{name}.parquet", index=False)
print("Tersimpan ke", OUTDIR, "->", list(splits))

# (Opsional, Colab) unduh hasil:
# from google.colab import files
# for name in splits: files.download(f"{OUTDIR}/{name}.parquet")

Output: `data/processed/indobert/{train,val,test}.parquet` (kolom **`bert`** = teks bersih).
**Lanjut:** fine-tuning IndoBERT (Colab/GPU) — unggah ketiga parquet ini.